# 📊 Notebook 01 — Exploratory Data Analysis
**Fraud Detection System** | Credit Card Fraud Dataset
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/raw/creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Basic Dataset Info

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicates ===')
print(f'Duplicate rows: {df.duplicated().sum()}')
print('\n=== Basic Statistics ===')
df[['Amount','Time']].describe()

## 2. Class Distribution (Fraud vs Normal)

In [ ]:
counts = df['Class'].value_counts()
print(counts.rename({0:'Normal', 1:'Fraud'}))
print(f'\nFraud %: {100*counts[1]/len(df):.3f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#3498db','#e74c3c']

axes[0].bar(['Normal','Fraud'], counts.values, color=colors, width=0.5)
axes[0].set_title('Transaction Class Counts', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, val in enumerate(counts.values):
    axes[0].text(i, val+500, f'{val:,}', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=['Normal','Fraud'], colors=colors,
            autopct='%1.2f%%', explode=(0,0.1), startangle=90)
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Transaction Amount Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, color, name in [(0,'#3498db','Normal'),(1,'#e74c3c','Fraud')]:
    sns.kdeplot(df[df['Class']==label]['Amount'], ax=axes[0],
                fill=True, alpha=0.4, color=color, label=name)
axes[0].set_title('Amount Distribution by Class', fontsize=12)
axes[0].set_xlabel('Amount (euros)')
axes[0].legend()

df_plot = df.copy()
df_plot['Label'] = df_plot['Class'].map({0:'Normal', 1:'Fraud'})
sns.boxplot(data=df_plot, x='Label', y='Amount',
            palette={'Normal':'#3498db','Fraud':'#e74c3c'}, ax=axes[1])
axes[1].set_title('Amount Box Plot', fontsize=12)

plt.tight_layout()
plt.show()

print(df.groupby('Class')['Amount'].describe().rename(index={0:'Normal',1:'Fraud'}))

## 4. Feature Distributions V1–V28

In [ ]:
v_cols = [f'V{i}' for i in range(1, 29)]
normal = df[df['Class']==0]
fraud  = df[df['Class']==1]

fig, axes = plt.subplots(7, 4, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(v_cols):
    sns.kdeplot(normal[col], ax=axes[i], color='#3498db', label='Normal', fill=True, alpha=0.3)
    sns.kdeplot(fraud[col],  ax=axes[i], color='#e74c3c', label='Fraud',  fill=True, alpha=0.3)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
    if i == 0:
        axes[i].legend(fontsize=7)

plt.suptitle('Feature Distributions: Normal vs Fraud', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation with Target

In [ ]:
corr = df.corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(5, 10))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr.values.reshape(-1,1),
            annot=True, fmt='.2f',
            yticklabels=corr.index,
            xticklabels=['Corr w/ Class'],
            cmap=cmap, center=0, ax=ax)
ax.set_title('Feature Correlations with Fraud (Class)', fontsize=12)
plt.tight_layout()
plt.show()